# Prepare reviewed FPV demonstrations

Reviewed human FPV baseline; no NVIDIA API calls, cloud runtime creation or robot commands. Set `CAFEROOMBA_REVISION` to the full published implementation commit SHA. Mount persistent storage and set `CAFEROOMBA_WORKSPACE` to the same location in notebooks 01–03. For consumer Colab, explicitly mount Drive if wanted: `from google.colab import drive; drive.mount('/content/drive')`. Enterprise storage/auth varies; use your persistent filesystem. Ordinary `/content` and `/tmp` are not durable across runtime deletion. Never paste keys into cells. See `docs/COLAB_TRAINING.md`.

In [ ]:
import os, re, subprocess, sys, shutil
from pathlib import Path
REPO_REVISION = os.environ.get("CAFEROOMBA_REVISION", "SET_IMPLEMENTATION_COMMIT_SHA")
REPO = Path(os.environ.get("CAFEROOMBA_REPO", "/content/caferoomba"))
if not re.fullmatch(r"[0-9a-f]{40}", REPO_REVISION):
    raise ValueError("Set CAFEROOMBA_REVISION to the full published implementation commit SHA.")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/EdwinKestler/caferoomba.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", REPO_REVISION], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if actual != REPO_REVISION:
    raise ValueError("Existing checkout differs; use a separate checkout without overwriting local work.")
if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip():
    raise ValueError("Use a clean checkout for reproducible source provenance.")
if sys.version_info < (3, 11):
    raise RuntimeError("Python 3.11 or newer required.")
# Preserve an existing CUDA-enabled Torch installation; do not force CPU-only wheels.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO) + "[cpu,dev]"], check=True)
sys.path.insert(0, str(REPO / "src"))
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    raise RuntimeError("Install ffmpeg/ffprobe before preparing videos.")
print("source revision:", actual)


In [ ]:
workspace_text = os.environ.get("CAFEROOMBA_WORKSPACE", "")
if not workspace_text:
    raise ValueError("Set CAFEROOMBA_WORKSPACE to your mounted persistent workspace.")
WORKSPACE = Path(workspace_text).resolve()
if not WORKSPACE.is_dir():
    raise ValueError("Workspace must already exist on persistent storage.")
DATASET = WORKSPACE / "dataset-v1"
RUN = WORKSPACE / "run-v1"
MANIFEST = DATASET / "manifest.json"


Place videos under `WORKSPACE/media/` and combined reviewed JSONL at `WORKSPACE/labels.reviewed.jsonl`. Video paths are relative to the media root. At least three independent runs/sessions are required. Copied videos do not count as independent. Complete human review; never mechanically mark unlabeled drafts reviewed. This step verifies hashes, selects causal frames from actual PTS, and publishes an immutable dataset.

In [ ]:
from caferoomba.data.prepare import prepare_dataset, load_prepared_dataset
if MANIFEST.exists():
    clips = load_prepared_dataset(MANIFEST)
    print("Verified existing dataset:", len(clips), "clips")
else:
    prepared = prepare_dataset(
        labels_path=WORKSPACE / "labels.reviewed.jsonl", media_root=WORKSPACE / "media",
        output_dir=DATASET, frame_count=8, period_ms=250, max_frame_age_ms=100,
    )
    print(prepared)


NVIDIA teacher suggestions remain optional and require a supported endpoint, bounded annotation plan and human review. They are not consumed by this baseline loss. Synthetic checks remain separate: `python -m caferoomba demo-fixture`.